# Module Review

Papermill template for `span.module_notebook_builder`. For every executable
shipped by the module's RPMs, look up the domains that have `file:entrypoint`
on the executable's SELinux type, then run `domain_summary` on each.

In [ ]:
module_name = ""
policy_path = ""
rpms = []
policy_modules = []
executables = []

In [ ]:
from IPython.display import display, Markdown

from span import load_policy

p = load_policy(policy_path)

In [ ]:
display(Markdown(f"# Module: `{module_name}`"))
display(Markdown(
    f"**RPMs:** {len(rpms)} &nbsp;&nbsp; "
    f"**Policy modules (.pp):** {len(policy_modules)} &nbsp;&nbsp; "
    f"**Executables:** {len(executables)}"
))

if rpms:
    display(Markdown("## RPMs\n\n" + "\n".join(f"- `{r}`" for r in rpms)))
if policy_modules:
    display(Markdown("## Policy modules\n\n" + "\n".join(f"- `{m}`" for m in policy_modules)))

In [ ]:
def domains_for_entrypoint(policy, se_type):
    rules = policy.terules_query_raw(
        target=se_type, tclass=["file"], perms=["entrypoint"]
    )
    return sorted({str(x) for r in rules for x in r.source.expand()})

In [ ]:
summarized = set()

entrypoint_execs = [e for e in executables if e.get("is_entrypoint")]
other_execs = [e for e in executables if not e.get("is_entrypoint")]

if not entrypoint_execs:
    display(Markdown("_No executables resolved to an entrypoint type._"))

for ep in entrypoint_execs:
    se_type = ep["entrypoint_type"]
    rpm_list = ", ".join(f"`{r}`" for r in ep.get("rpms", []))
    display(Markdown(
        f"## `{ep['path']}`\n\n"
        f"**Entrypoint type:** `{se_type}` &nbsp;&nbsp; "
        f"**From RPM(s):** {rpm_list or '_unknown_'}"
    ))

    domains = domains_for_entrypoint(p, se_type)
    if not domains:
        display(Markdown(f"_No domains have `file:entrypoint` on `{se_type}`._"))
        continue

    display(Markdown("**Domains:** " + ", ".join(f"`{d}`" for d in domains)))
    for d in domains:
        if d in summarized:
            display(Markdown(f"_Domain `{d}` summarized above; skipping._"))
            continue
        summarized.add(d)
        p.domain_summary(d)

In [ ]:
if other_execs:
    lines = ["## Other executables (no entrypoint type)\n"]
    for ep in other_execs:
        t = ep.get("entrypoint_type") or "_no SELinux type matched_"
        lines.append(f"- `{ep['path']}` -> `{t}`")
    display(Markdown("\n".join(lines)))